In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql.functions import col, current_timestamp, lit
from etl.transformations.common import create_spark, read_raw_table, write_clickhouse

MINIO_STAGING_BUCKET = os.environ.get("MINIO_STAGING_BUCKET", "staging")

In [2]:
def main():
    spark = create_spark("load_dim_crop")

    crops_df = read_raw_table(
        spark,
        MINIO_STAGING_BUCKET,
        "crops",
    )

    categories_df = read_raw_table(
        spark,
        MINIO_STAGING_BUCKET,
        "crop_categories",
    )

    crops_df.show()
    categories_df.show()

    dim_crop_df = crops_df.join(
        categories_df,
        crops_df.category_id == categories_df.id,
        "left",
    ).select(
        crops_df.id.alias("crop_id"),
        crops_df.name,
        crops_df.description,
        crops_df.category_id.alias("crop_category_id"),
        categories_df.name.alias("category_name"),
        lit(0).alias("is_high_value"),
        current_timestamp().alias("_loaded_at"),
    )

    dim_crop_df.count()
    dim_crop_df.show(25)
    dim_crop_df.printSchema()

    print("Rows before write:", dim_crop_df.count())

    dim_crop_df.select(
        "crop_id",
        "name",
        "crop_category_id",
        "category_name",
    ).show(25, truncate=False)

    write_clickhouse(
        dim_crop_df,
        "dim_crop",
    )

    spark.stop()

In [3]:
if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/20 16:47:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/20 16:47:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+---+-----------+------------------+--------------------+----------+----------+
| id|category_id|              name|         description|created_at|updated_at|
+---+-----------+------------------+--------------------+----------+----------+
|  1|          2|             Basil|Aromatic herb use...|1784565399|1784565399|
|  2|          2|              Mint|Refreshing herb u...|1784565399|1784565399|
|  3|          2|           Parsley|Versatile herb fo...|1784565399|1784565399|
|  4|          2|          Cilantro|Herb with citrusy...|1784565399|1784565399|
|  5|          2|             Thyme|Small-leaved herb...|1784565399|1784565399|
|  6|          2|          Rosemary|Fragrant, woody h...|1784565399|1784565399|
|  7|          1|   Romaine Lettuce|Crisp, elongated ...|1784565399|1784565399|
|  8|          1|Butterhead Lettuce|Soft, tender leav...|1784565399|1784565399|
|  9|          1|   Arugula Lettuce|Peppery, tangy gr...|1784565399|1784565399|
| 10|          1|           Spinach|Tend

26/07/20 16:47:22 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
